    "# DINO SDK v2.7.1 - Azure SQL Logging (CORRIGIDO)
",
    "
",
    "Este notebook demonstra como usar o DINO SDK v2.7.1 com logging no Azure SQL Database ao invés do Unity Catalog.
",
    "
",
    "## 🆕 **Novidades da v2.7.1:**
",
    "- ✅ **CORREÇÃO:** Removido campo `created_at` para compatibilidade com Azure SQL
",
    "- ✅ Suporte completo ao Azure SQL Database para logging
",
    "- ✅ Compatibilidade com Unity Catalog (modo padrão)
",
    "- ✅ Configuração flexível via parâmetros
",
    "- ✅ Todos os campos de log preservados
",
    "- ✅ Fallback automático para logging local"

    "## 1. Instalação do DINO SDK v2.7.1
",
    "
",
    "Instale o wheel gerado:"

In [ ]:
    "# Instalar o wheel do DINO SDK v2.7.1
",
    "%pip install /path/to/dino_sdk-2.7.1-py3-none-any.whl --force-reinstall"

## 2. Configuração dos Parâmetros Azure SQL

Configure as variáveis de conexão com o Azure SQL:

In [ ]:
# Configurações Azure SQL (forneça seus valores)
azure_sql_server = "data-master-dev-sql-9873"
azure_sql_database = "data-master-dev-db-logs"
azure_sql_username = "sqladmin"
azure_sql_password = "v:GJKj?}p@F@lEHp"
azure_sql_table = "dbo.dino_ingestion_logs"  # Nome da tabela (opcional, default: dbo.dino_ingestion_logs)

print("✅ Configurações Azure SQL definidas")
print(f"🔗 Servidor: {azure_sql_server}.database.windows.net")
print(f"📊 Database: {azure_sql_database}")
print(f"👤 Usuário: {azure_sql_username}")
print(f"🗂️ Tabela: {azure_sql_table}")

## 3. Inicialização do DINO SDK com Azure SQL

Inicialize o IngestionEngine com os parâmetros Azure SQL:

In [ ]:
from dino_sdk import IngestionEngine, IngestionConfig
from pyspark.sql import SparkSession

# Obter sessão Spark ativa
spark = SparkSession.getActiveSession()

# Inicializar IngestionEngine com Azure SQL
engine = IngestionEngine(
    spark=spark,
    azure_sql_server=azure_sql_server,
    azure_sql_database=azure_sql_database, 
    azure_sql_username=azure_sql_username,
    azure_sql_password=azure_sql_password,
    azure_sql_table=azure_sql_table
)

print("🦕 DINO SDK v2.7.1 inicializado com Azure SQL!")

## 4. Configuração da Ingestão

Configure os parâmetros de ingestão normalmente:

In [ ]:
# Configuração da ingestão (seus parâmetros habituais)
config = IngestionConfig(
    catalog_name="data_master_dev_dbw",
    schema_name="bronze", 
    table_name="resultados_2024",
    source_path="/Volumes/data_master_dev_dbw/bronze/raw/resultados_2024/",
    file_extension="csv",
    type_run="batch",
    delimiter=";",
    has_header=True,
    infer_schema=False,
    enable_liquid_clustering=True
)

print("⚙️ Configuração de ingestão definida")
print(f"📊 Tabela destino: {config.catalog_name}.{config.schema_name}.{config.table_name}")
print(f"📁 Source path: {config.source_path}")
print(f"🔄 Tipo de execução: {config.type_run}")

## 5. Execução da Ingestão

Execute a ingestão normalmente. O logging será automaticamente gravado no Azure SQL:

In [ ]:
# Executar ingestão
resultado = engine.ingest(config)

print("\n🎯 Resultado da Ingestão:")
print(f"✅ Status: {resultado.get('status')}")
print(f"📋 Execution ID: {resultado.get('execution_id')}")
print(f"⏱️ Duração: {resultado.get('execution_time_seconds', 0):.2f}s")
print(f"📊 Registros processados: {resultado.get('records_processed', 0)}")

## 6. Verificação dos Logs no Azure SQL

Verifique se os logs foram gravados corretamente no Azure SQL:

In [ ]:
# Verificar logs no Azure SQL
jdbc_url = f"jdbc:sqlserver://{azure_sql_server}.database.windows.net:1433;database={azure_sql_database}"
connection_properties = {
    "user": azure_sql_username,
    "password": azure_sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Ler logs da tabela Azure SQL
logs_df = spark.read.jdbc(
    url=jdbc_url, 
    table=azure_sql_table, 
    properties=connection_properties
)

# Mostrar últimos logs
print("📋 Últimos logs gravados no Azure SQL:")
logs_df.orderBy("start_time", ascending=False).limit(5).show(truncate=False)

## 7. Campos Disponíveis no Log

Veja todos os campos que são gravados na tabela de log:

In [ ]:
# Mostrar schema da tabela de logs
print("📊 Campos disponíveis na tabela de logs:")
logs_df.printSchema()

# Mostrar último log detalhado
ultimo_log = logs_df.orderBy("start_time", ascending=False).limit(1)
print("\n🔍 Último log detalhado:")
ultimo_log.show(1, truncate=False, vertical=True)

## 📋 **Resumo dos Parâmetros**

Para usar o DINO SDK v2.7.1 com Azure SQL, passe os seguintes parâmetros ao inicializar o `IngestionEngine`:

```python
engine = IngestionEngine(
    spark=spark,                              # Obrigatório
    azure_sql_server="data-master-dev-sql-9873",    # Nome do servidor (sem .database.windows.net)
    azure_sql_database="data-master-dev-db-logs",   # Nome do database 
    azure_sql_username="sqladmin",                  # Usuário
    azure_sql_password="v:GJKj?}p@F@lEHp",          # Senha
    azure_sql_table="dbo.dino_ingestion_logs"       # Opcional, default: dbo.dino_ingestion_logs
)
```

## ✅ **Benefícios:**
- 🔗 Logs centralizados no Azure SQL
- 📊 Todos os campos preservados (registros lidos/gravados, duração, etc.)
- 🔄 Compatibilidade com Unity Catalog (omita parâmetros Azure SQL)
- 🛡️ Fallback automático para logging local em caso de erro
- 🚀 Mesma interface de uso, apenas configuração diferente